In [0]:

%pip install xgboost scikit-learn seaborn matplotlib
dbutils.library.restartPython()



In [0]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from datetime import date
from dateutil.relativedelta import relativedelta
from functools import reduce

# Spark (processamento distribuído — sem toPandas)
from pyspark.sql import functions as F
from pyspark.sql import Window
from pyspark.sql.types import StringType


pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore")

In [0]:

# Parâmetros automáticos — calculados a partir da data de execução
# O notebook roda no dia 05 de cada mês; dados disponíveis até o mês anterior.
# Previsões: mês seguinte (M+1), M+2 e M+3.
from datetime import date
from dateutil.relativedelta import relativedelta
_hoje = date.today()
_primeiro_dia_mes = _hoje.replace(day=1)

HORIZONTES = []
for offset in range(1, 4):  # offset 1 → M+1, 2 → M+2, 3 → M+3
    _alvo = _primeiro_dia_mes + relativedelta(months=offset)
    HORIZONTES.append({
        "lag_inicio": offset + 1,           # M+1 → lag 2, M+2 → lag 3, M+3 → lag 4
        "lag_final":  offset + 1 + 10,      # sempre 11 lags (ex: 2-12, 3-13, 4-14)
        "mes_ano":    int(_alvo.strftime("%Y%m")),
        "mes":        _alvo.strftime("%m"),
        "ano":        _alvo.year,
        "trimestre":  (_alvo.month - 1) // 3 + 1,
    })

# Mês de validação = mês anterior ao atual (último mês completo com dados)
_valid_date = _primeiro_dia_mes - relativedelta(months=1)
VALID = int(_valid_date.strftime("%Y%m"))

# Log de conferência
print(f"Data de execução: {_hoje}")
print(f"Mês de validação (VALID): {VALID}")
for h in HORIZONTES:
    print(f"  Horizonte {h['mes']}/{h['ano']}  →  lag {h['lag_inicio']} a {h['lag_final']}")

In [0]:

# Primeiro dia do mês anterior e 2 anos antes dele
df_pedidos = spark.sql("""
  SELECT cd_material, cd_emissor, qtd_convertido_total_ordem, cd_ano_mes_criacao, dt_criacao_doc, cd_centro
  FROM desenvolvimento.gold_ciex.tb_sap_zsd291
  WHERE dt_criacao_doc >= add_months(trunc(add_months(current_date(), -1), 'MM'), -36)
    AND dt_criacao_doc < trunc(add_months(current_date(), -1), 'MM')
""")
 
dados_pedidos = df_pedidos  # Spark DataFrame (sem toPandas)

In [0]:
df_faturamento = spark.sql("""SELECT id_cliente, qtd_faturada_convertida, id_sku, nr_mes_faturamento, nr_ano_faturamento, dt_faturamento, cd_centro FROM desenvolvimento.gold_ciex.tb_sap_zsd097n WHERE dt_faturamento >= add_months(trunc(add_months(current_date(), -1), 'MM'), -48)
    AND dt_faturamento < trunc(current_date(), 'MM')
""")
dados_faturamento = df_faturamento  # Spark DataFrame (sem toPandas)

In [0]:
# Replica 2025-08 em 36/48 meses para que o pipeline tenha histórico suficiente
meses_ped = sorted([(date(2026, 9, 1) - relativedelta(months=i)).strftime("%Y-%m") for i in range(36)])
meses_fat = sorted([(date(2026, 9, 1) - relativedelta(months=i)).strftime("%Y-%m") for i in range(48)])
ref_ped = df_pedidos.filter(F.col("cd_ano_mes_criacao").cast("string").contains("2025-08"))
dados_pedidos = (ref_ped.crossJoin(spark.createDataFrame([(m,) for m in meses_ped], ["novo_mes"]))
    .withColumn("cd_ano_mes_criacao", F.col("novo_mes"))
    .withColumn("dt_criacao_doc", F.to_date(F.concat(F.col("novo_mes"), F.lit("-01")), "yyyy-MM-dd"))
    .drop("novo_mes"))
ref_fat = df_faturamento.filter((F.col("nr_ano_faturamento") == 2025) & (F.col("nr_mes_faturamento") == 8))
dados_faturamento = (ref_fat.crossJoin(spark.createDataFrame([(m,) for m in meses_fat], ["novo_mes"]))
    .withColumn("nr_ano_faturamento", F.split(F.col("novo_mes"), "-").getItem(0).cast("int"))
    .withColumn("nr_mes_faturamento", F.split(F.col("novo_mes"), "-").getItem(1).cast("int"))
    .withColumn("dt_faturamento", F.to_date(F.concat(F.col("novo_mes"), F.lit("-01")), "yyyy-MM-dd"))
    .drop("novo_mes"))
print(f"Pedidos: {ref_ped.count():,} linhas x 36 meses | Faturamento: {ref_fat.count():,} linhas x 48 meses")

In [0]:
df_cliente = spark.sql("""SELECT desc_escritorio_vendas AS BR, desc_equipe_vendas AS GR, nome_segmento AS Sub_Canal, id_cliente AS Cliente FROM desenvolvimento.gold_corporativo.cliente""")
cliente = df_cliente.withColumn("Cliente", F.lpad(F.col("Cliente").cast("string"), 10, "0"))

In [0]:
df_produto = spark.sql("""SELECT id_sku AS SKU, id_sku2 AS SKU2, familia AS Familia FROM desenvolvimento.gold_corporativo.produto""")
produto = df_produto

In [0]:

display(dados_pedidos.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in dados_pedidos.columns]))

In [0]:

display(dados_faturamento.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in dados_faturamento.columns]))

In [0]:
# Preencher nulos de qtd_convertido_total_ordem com 0 e remover linhas com nulos nas demais colunas
dados_pedidos = (dados_pedidos
    .withColumn("qtd_convertido_total_ordem", F.coalesce(F.col("qtd_convertido_total_ordem").cast("double"), F.lit(0.0)))
    .dropna())

# Preencher nulos de qtd_faturada_convertida com 0 e remover linhas com nulos nas demais colunas
dados_faturamento = (dados_faturamento
    .withColumn("qtd_faturada_convertida", F.coalesce(F.col("qtd_faturada_convertida").cast("double"), F.lit(0.0)))
    .dropna())

In [0]:
# Substituir valores negativos por 0
dados_faturamento = dados_faturamento.withColumn("qtd_faturada_convertida", F.greatest(F.col("qtd_faturada_convertida"), F.lit(0.0)))
dados_pedidos = dados_pedidos.withColumn("qtd_convertido_total_ordem", F.greatest(F.col("qtd_convertido_total_ordem"), F.lit(0.0)))

In [0]:
# tratar nomes das tabelas de pedidos e faturamento
dados_faturamento = dados_faturamento.withColumnsRenamed({"id_cliente": "Cliente", "id_sku": "SKU", "nr_mes_faturamento": "Mes", "nr_ano_faturamento": "Ano", "dt_faturamento": "Data", "cd_centro": "Centro", "qtd_faturada_convertida": "Faturamento"})
dados_pedidos = dados_pedidos.withColumnsRenamed({"cd_material": "SKU", "cd_emissor": "Cliente", "qtd_convertido_total_ordem": "Pedidos", "cd_centro": "Centro", "dt_criacao_doc": "Data", "cd_ano_mes_criacao": "Ano_Mes"})

In [0]:
display(cliente)

In [0]:
# ajustar data faturamento
dados_faturamento = (dados_faturamento
    .withColumn("Ano", F.col("Ano").cast("int"))
    .withColumn("Mes", F.col("Mes").cast("int"))
    .withColumn("Data", F.col("Ano") * 100 + F.col("Mes"))  # AAAAMM inteiro
    .groupBy('Cliente', 'SKU', 'Mes', 'Ano', 'Data', 'Centro')
    .agg(F.sum("Faturamento").alias("Faturamento")))
dados_crescimento = dados_faturamento


In [0]:
# manter apenas os 24 meses mais recentes (2 anos)
top_36 = dados_faturamento.select("Data").distinct().orderBy(F.desc("Data")).limit(36)
dados_faturamento = dados_faturamento.join(F.broadcast(top_36), on="Data", how="left_semi")

In [0]:
# ajustar data pedidos
dados_pedidos = (dados_pedidos
    .withColumn("Mes", F.month("Data"))
    .withColumn("Ano", F.year("Data"))
    .withColumn("Data", F.col("Ano") * 100 + F.col("Mes"))  # AAAAMM inteiro
    .groupBy('Cliente', 'SKU', 'Mes', 'Ano', 'Data', 'Centro')
    .agg(F.sum("Pedidos").alias("Pedidos")))

In [0]:
# unir a faturamento e pedidos
chaves = ["Cliente", "SKU", "Data", "Centro", "Mes", "Ano"]
dados_brasil = dados_pedidos.join(dados_faturamento, on=chaves, how="outer")

In [0]:
# juntar com cliente e produto
dados_brasil = dados_brasil.join(cliente, on="Cliente", how="left").join(produto, on="SKU", how="left")
dados_crescimento = dados_crescimento.join(cliente, on="Cliente", how="left").join(produto, on="SKU", how="left")

In [0]:
# agregar
# (groupby do pandas descarta chaves nulas e soma nulos como 0 — mesmo comportamento abaixo)
chaves_agg = ["BR", "GR", "Sub_Canal", "SKU", "Centro", "Familia", "Ano", "Mes", "Data"]
dados_brasil = (dados_brasil
    .dropna(subset=chaves_agg)
    .groupBy(*chaves_agg)
    .agg(
        F.coalesce(F.sum("Pedidos"), F.lit(0.0)).alias("Pedidos"),
        F.coalesce(F.sum("Faturamento"), F.lit(0.0)).alias("Faturamento")
    ))
dados_crescimento = (dados_crescimento
    .groupBy("Data", "Ano", "Mes")
    .agg(F.coalesce(F.sum("Faturamento"), F.lit(0.0)).alias("Faturamento")))

In [0]:
# criar colunas necessárias (trimestre)
dados_brasil = dados_brasil.withColumn("Trimestre", F.floor((F.col("Mes") - 1) / 3).cast("int") + 1)
dados_crescimento = dados_crescimento.withColumn("Trimestre", F.floor((F.col("Mes") - 1) / 3).cast("int") + 1)


In [0]:

# from pyspark.sql.functions import col
# # teste
# df_dados = spark.sql('''select Sub_Canal, BR, GR, ID_Produto_Ajustado AS SKU, `ID_Localização` AS Centro, Familia, Ano, Mes, ano_mes AS Data, Pedidos, Faturamento, Trimestre from desenvolvimento.bronze_demanda.tb_resultado''')
# dados_brasil = df_dados  # Spark DataFrame (sem toPandas)
# dados_brasil = dados_brasil.withColumn("Mes", col("Mes").cast("int"))



In [0]:
display(dados_brasil.filter(F.col("Data") == 202604))

In [0]:

# # Parâmetros automáticos — calculados a partir da data de execução
# # O notebook roda no dia 05 de cada mês; dados disponíveis até o mês anterior.
# # Previsões: mês seguinte (M+1), M+2 e M+3.

# _hoje = date(2026, 4, 5)  # Fixado em abril de 2026
# _primeiro_dia_mes = _hoje.replace(day=1)

# HORIZONTES = []
# for offset in range(1, 4):  # offset 1 → M+1, 2 → M+2, 3 → M+3
#     _alvo = _primeiro_dia_mes + relativedelta(months=offset)
#     HORIZONTES.append({
#         "lag_inicio": offset + 1,           # M+1 → lag 2, M+2 → lag 3, M+3 → lag 4
#         "lag_final":  offset + 1 + 10,      # sempre 11 lags (ex: 2-12, 3-13, 4-14)
#         "mes_ano":    int(_alvo.strftime("%Y%m")),
#         "mes":        _alvo.strftime("%m"),
#         "ano":        _alvo.year,
#         "trimestre":  (_alvo.month - 1) // 3 + 1,
#     })

# # Mês de validação = mês anterior ao atual (último mês completo com dados)
# _valid_date = _primeiro_dia_mes - relativedelta(months=1)
# VALID = int(_valid_date.strftime("%Y%m"))

# # Log de conferência
# print(f"Data de execução: {_hoje}")
# print(f"Mês de validação (VALID): {VALID}")
# for h in HORIZONTES:
#     print(f"  Horizonte {h['mes']}/{h['ano']}  →  lag {h['lag_inicio']} a {h['lag_final']}")

In [0]:
# Identificar o ano_mes mais recente nos dados históricos
ano_mes_recente = dados_brasil.agg(F.max("Data")).first()[0]
print(f"ano_mes mais recente: {ano_mes_recente}")

# Filtrar dados_brasil pela data mais recente
dados_mes_recente = dados_brasil.filter(F.col("Data") == ano_mes_recente)

# Tabela (pequena) com os 3 horizontes de previsão
horizontes_df = spark.createDataFrame(
    [(int(h["mes"]), int(h["ano"]), int(h["mes_ano"]), int(h["trimestre"])) for h in HORIZONTES],
    "Mes int, Ano int, Data int, Trimestre int"
)

# Replicar o mês mais recente para cada horizonte (cross join com broadcast)
placeholders_futuros = (dados_mes_recente
    .drop("Mes", "Ano", "Data", "Trimestre")
    .crossJoin(F.broadcast(horizontes_df))
    .select(*dados_brasil.columns))



In [0]:
# Unir dados_brasil com placeholders dos 3 meses futuros
dados_brasil = dados_brasil.unionByName(placeholders_futuros)

# Agregar placeholders no nível de dados_crescimento (Data/Ano/Mes)
placeholders_cresc = (placeholders_futuros
    .groupBy("Data", "Ano", "Mes")
    .agg(F.sum("Faturamento").cast("double").alias("Faturamento"))
    .withColumn("Trimestre", F.floor((F.col("Mes") - 1) / 3).cast("int") + 1))
dados_crescimento = (dados_crescimento
    .withColumn("Faturamento", F.col("Faturamento").cast("double"))
    .unionByName(placeholders_cresc))

dados_brasil = dados_brasil.withColumns({
    "Pedidos": F.col("Pedidos").cast("double"),
    "Faturamento": F.col("Faturamento").cast("double"),
})


In [0]:
# criar taxa crescimento
# Para cada mês, calcular a soma do faturamento dos 2 anos mais recentes
# dividida pela média anual dos 2 anos anteriores a esses.
# Ex: taxa jan/2027 = (fat jan/2025 + fat jan/2026) / (média mensal 2024 + média mensal 2025)

# Remover taxa anterior caso a célula seja re-executada
if "taxa" in dados_crescimento.columns:
    dados_crescimento = dados_crescimento.drop("taxa")

# Agregar faturamento por Ano e Mes
cresc_mensal = dados_crescimento.groupBy("Ano", "Mes").agg(F.sum("Faturamento").alias("Faturamento"))

# 1) Média mensal por ano (total do ano / nº de meses no ano)
media_anual = cresc_mensal.groupBy("Ano").agg(
    (F.sum("Faturamento") / F.count(F.lit(1))).alias("Media_Mensal_Ano")
)

# 2) taxa(mes, ano) = (fat(mes, ano-2) + fat(mes, ano-1)) / (media_mensal(ano-3) + media_mensal(ano-2))
#    Implementado com joins deslocando o Ano (sem loops no driver)
fat_m1 = cresc_mensal.select((F.col("Ano") + 1).alias("Ano"), "Mes", F.col("Faturamento").alias("fat_ano_m1"))
fat_m2 = cresc_mensal.select((F.col("Ano") + 2).alias("Ano"), "Mes", F.col("Faturamento").alias("fat_ano_m2"))
med_m2 = media_anual.select((F.col("Ano") + 2).alias("Ano"), F.col("Media_Mensal_Ano").alias("media_m2"))
med_m3 = media_anual.select((F.col("Ano") + 3).alias("Ano"), F.col("Media_Mensal_Ano").alias("media_m3"))

taxa_df = (cresc_mensal.select("Ano", "Mes")
    .join(fat_m1, ["Ano", "Mes"], "left")
    .join(fat_m2, ["Ano", "Mes"], "left")
    .join(med_m2, ["Ano"], "left")
    .join(med_m3, ["Ano"], "left")
    .withColumn("numerador", F.col("fat_ano_m2") + F.col("fat_ano_m1"))
    .withColumn("denominador", F.col("media_m3") + F.col("media_m2"))
    # nulo quando faltar algum ano ou denominador = 0 (equivalente ao np.nan anterior)
    .withColumn("taxa", F.when(F.col("denominador") != 0, F.col("numerador") / F.col("denominador")))
    .select("Ano", "Mes", "taxa"))

# Join de volta no dados_crescimento
dados_crescimento = dados_crescimento.join(taxa_df, on=["Ano", "Mes"], how="left")



In [0]:
display(dados_crescimento.orderBy("Data"))
# conferir taxa

In [0]:
dados_crescimento = dados_crescimento.select("Ano", "Data", "Mes", "taxa", "Trimestre")

In [0]:
# Calcular média anual por grupo
group_cols = ["BR", "GR", "Sub_Canal", "Centro", "SKU", "Ano", "Familia"]
dados_media = dados_brasil.groupBy(*group_cols).agg(F.avg("Faturamento").alias("Media_ano"))

# Criar lag da média ano (ano anterior) por grupo — window distribuída
group_cols_lag = ["BR", "GR", "Sub_Canal", "Centro", "SKU", "Familia"]
w_media = Window.partitionBy(*group_cols_lag).orderBy("Ano")
dados_media = dados_media.withColumn("Media_ano", F.lag("Media_ano", 1).over(w_media))

In [0]:
# unir dados_brasil e crescimento
dados_brasil = dados_brasil.join(F.broadcast(dados_crescimento), on=["Data", "Ano", "Mes", "Trimestre"], how="left")
# unir dados_brasil e media
dados_brasil = dados_brasil.join(dados_media, on=["BR", "GR", "Sub_Canal", "Centro", "SKU", "Familia", "Ano"], how="left")
dados_brasil = dados_brasil.withColumn("Projecao", F.col("Media_ano") * F.col("taxa"))

In [0]:
display(dados_brasil)

In [0]:
display(dados_brasil.select([F.count(F.when(F.col(c).isNull(), 1)).alias(c) for c in dados_brasil.columns]))

In [0]:
# dados_brasil = dados_brasil.dropna()

In [0]:
mes_pt_en = {
    1: "JAN", 2: "FEV", 3: "MAR", 4: "ABR", 5: "MAI", 6: "JUN",
    7: "JUL", 8: "AGO", 9: "SET", 10: "OUT", 11: "NOV", 12: "DEZ"
}
mes_map = F.create_map(*[F.lit(x) for k, v in mes_pt_en.items() for x in (k, v)])
dados_brasil = dados_brasil.withColumn("Mes", F.try_element_at(mes_map, F.col("Mes").cast("int")))

# Materializar a base (evita recomputar leitura + joins a cada ação do loop de modelos)
dados_brasil = dados_brasil.localCheckpoint()
print(f"dados_brasil: {dados_brasil.count():,} linhas")


In [0]:
# ============================================================================
# Função reutilizável: processa um horizonte completo
# (lag features → RF zero-classificador → XGBoost distribuído → métricas)
# ============================================================================
import time as _time
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StringIndexerModel
from pyspark.ml.classification import RandomForestClassifier as SparkRFC
from xgboost.spark import SparkXGBRegressor

DIV = 2
CHAVES_LAG = ["BR", "GR", "Sub_Canal", "Centro", "SKU", "Familia"]
W_LAG = Window.partitionBy(*CHAVES_LAG).orderBy("Data")

def _signed_log1p(c):
    return F.signum(c) * F.log1p(F.abs(c))

def _log1p_pos(c):
    return F.log1p(F.greatest(c, F.lit(0.0)))

def _media_cols(cols):
    return reduce(lambda a, b: a + b, [F.col(c) for c in cols]) / F.lit(float(len(cols)))


def processar_horizonte(idx, horizonte, dados_base, mes_map_col, valid_val,
                        num_workers_xgb=4, verbose=True):
    """Processa um horizonte. Retorna (resultado_df, dict_metricas)."""
    t0 = _time.time()
    tag = f"H{idx+1}"
    def _p(msg):
        if verbose: print(f"[{tag}] {msg}")

    LAG_I   = horizonte["lag_inicio"]
    LAG_F   = horizonte["lag_final"]
    MES_ANO = horizonte["mes_ano"]
    _p(f"{horizonte['mes']}/{horizonte['ano']} — lag {LAG_I}..{LAG_F}")

    # 1 ── FEATURES LAG ──────────────────────────────────────────
    _p("1/7 features lag")
    dh = dados_base.withColumn("Mes", F.coalesce(F.col("Mes"),
        F.try_element_at(mes_map_col, (F.col("Data") % 100).cast("int"))))
    lags = {}
    for i in range(LAG_I, LAG_F + 1):
        lags[f"lag_{i}_Pedidos"]     = F.lag("Pedidos", i).over(W_LAG)
        lags[f"lag_{i}_Faturamento"] = F.lag("Faturamento", i).over(W_LAG)
    dh = dh.withColumns(lags)
    p6 = [f"lag_{i}_Pedidos"     for i in range(LAG_I, LAG_I+6)]
    p3 = [f"lag_{i}_Pedidos"     for i in range(LAG_I, LAG_I+3)]
    f6 = [f"lag_{i}_Faturamento" for i in range(LAG_I, LAG_I+6)]
    f3 = [f"lag_{i}_Faturamento" for i in range(LAG_I, LAG_I+3)]
    dh = dh.withColumns({
        "media_volume_6m_Pedidos":     _media_cols(p6),
        "media_volume_3m_Pedidos":     _media_cols(p3),
        "media_volume_6m_Faturamento": _media_cols(f6),
        "media_volume_3m_Faturamento": _media_cols(f3),
        "diff_1m_Pedidos":     F.col(f"lag_{LAG_I}_Pedidos")     - F.col(f"lag_{LAG_I+1}_Pedidos"),
        "diff_1m_Faturamento": F.col(f"lag_{LAG_I}_Faturamento") - F.col(f"lag_{LAG_I+1}_Faturamento"),
        "min_3m_Pedidos": F.least(*p3),  "max_3m_Pedidos": F.greatest(*p3),
        "min_3m_Faturamento": F.least(*f3), "max_3m_Faturamento": F.greatest(*f3),
    })
    dh = dh.dropna()

    # 2 ── LOG ──────────────────────────────────────────────────
    _p("2/7 transformação log")
    lpc = ["media_volume_6m_Pedidos","media_volume_3m_Pedidos",
           "media_volume_6m_Faturamento","media_volume_3m_Faturamento",
           "min_3m_Pedidos","max_3m_Pedidos","min_3m_Faturamento","max_3m_Faturamento"]
    dh = dh.withColumns({
        "Pedidos": F.log1p("Pedidos"), "Faturamento": F.log1p("Faturamento"),
        **{c: _log1p_pos(F.col(c)) for c in lpc},
        "diff_1m_Pedidos": _signed_log1p(F.col("diff_1m_Pedidos")),
        "diff_1m_Faturamento": _signed_log1p(F.col("diff_1m_Faturamento")),
    })
    dh = dh.localCheckpoint()
    bcols  = dh.columns
    ccols  = [x.name for x in dh.schema.fields if isinstance(x.dataType, StringType)]
    fcols  = [c for c in bcols if c != "Faturamento"]
    ncols  = [c for c in fcols if c not in ccols]

    # 3 ── CLASSIFICAÇÃO ZERO (RF) ─────────────────────────────────
    _p("3/7 classificador zero")
    tr_clf = (dh.filter(F.col("Data") < MES_ANO)
        .withColumn("zero", F.when(F.col("Faturamento") == 0, "Zero").otherwise("Não zero"))
        .select(*fcols, "zero"))
    te = dh.filter(F.col("Data") == MES_ANO)
    ixs = [StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep") for c in ccols]
    li  = StringIndexer(inputCol="zero", outputCol="label", handleInvalid="keep")
    va  = VectorAssembler(inputCols=[f"{c}_idx" for c in ccols]+ncols, outputCol="features", handleInvalid="keep")
    rfc = SparkRFC(featuresCol="features", labelCol="label", numTrees=300, seed=42, maxBins=512)
    clf = SparkPipeline(stages=ixs+[li, va, rfc]).fit(tr_clf)
    lm  = [s for s in clf.stages if isinstance(s, StringIndexerModel) and s.getOutputCol()=="label"][0]
    la  = F.array(*[F.lit(l) for l in lm.labels])
    tp  = (clf.transform(te).withColumn("zero", F.element_at(la, F.col("prediction").cast("int")+1))
           .select(*bcols, "zero").localCheckpoint())
    zdf = tp.filter(F.col("zero")=="Zero").drop("zero").withColumn("previsoes", F.lit(0.0))
    tnz = tp.filter(F.col("zero")!="Zero").drop("zero")

    # 4 ── XGBOOST DISTRIBUÍDO ────────────────────────────────────
    _p(f"4/7 XGBoost ({num_workers_xgb} workers)")
    txgb = (dh.filter((F.col("Data")<=valid_val) & (F.col("Faturamento")!=0))
        .withColumn("is_valid", F.col("Data")==valid_val))
    aa = txgb.filter(~F.col("is_valid"))
    enc = SparkPipeline(stages=[StringIndexer(inputCol=c, outputCol=f"{c}_cod",
        handleInvalid="keep", stringOrderType="alphabetAsc") for c in ccols]).fit(aa)
    nl = {s.getInputCol(): len(s.labels) for s in enc.stages}
    xin = [f"{c}_cod" if c in ccols else c for c in fcols]
    axgb = VectorAssembler(inputCols=xin, outputCol="fxgb", handleInvalid="keep")
    def _prep(df):
        df = enc.transform(df)
        df = df.withColumns({f"{c}_cod": F.when(F.col(f"{c}_cod")>=nl[c], F.lit(-1.0)).otherwise(F.col(f"{c}_cod")) for c in ccols})
        return axgb.transform(df)
    xr = SparkXGBRegressor(features_col="fxgb", label_col="Faturamento", prediction_col="pred_log",
        validation_indicator_col="is_valid", num_workers=num_workers_xgb,
        objective="reg:absoluteerror", eval_metric="mae",
        learning_rate=0.07928672, max_depth=8, subsample=0.3779839,
        colsample_bytree=0.4246809, min_child_weight=11, gamma=1.422943,
        random_state=42, n_estimators=989, early_stopping_rounds=100)
    mdl = xr.fit(_prep(txgb))

    # 5 ── MÉTRICAS VALIDAÇÃO ─────────────────────────────────────
    _p("5/7 métricas validação")
    pt = mdl.transform(_prep(txgb)).select("Faturamento","pred_log","is_valid").localCheckpoint()
    s2 = pt.filter(~F.col("is_valid")).agg(F.var_pop(F.col("Faturamento")-F.col("pred_log"))).first()[0]
    aj = float(s2) / DIV
    mv = (pt.filter(F.col("is_valid"))
        .select(F.expm1(F.col("pred_log")+F.lit(aj)).alias("yp"), F.expm1("Faturamento").alias("yv"))
        .agg(F.sqrt(F.avg(F.pow(F.col("yv")-F.col("yp"),2))).alias("rmse"),
             F.avg(F.abs(F.col("yv")-F.col("yp"))).alias("mae"),
             F.sum(F.pow(F.col("yv")-F.col("yp"),2)).alias("sse"),
             (F.var_pop("yv")*F.count(F.lit(1))).alias("sst"),
             F.sum("yv").alias("fat_real"),      # real do mês VALID (dados verdadeiros)
             F.sum("yp").alias("fat_prev"),      # previsto para o mês VALID
             F.count(F.lit(1)).alias("n")).first())
    r2v = 1 - mv["sse"]/mv["sst"] if mv["sst"] else float("nan")

    # 6 ── PREDIÇÃO ──────────────────────────────────────────────
    _p("6/7 previsão")
    pnz = (mdl.transform(_prep(tnz))
        .withColumn("previsoes", F.expm1(F.col("pred_log")+F.lit(aj)))
        .select(*bcols, "previsoes"))
    res = (pnz.unionByName(zdf.select(*bcols, "previsoes"))
        .withColumn("Faturamento", F.expm1("Faturamento"))
        .withColumns({"horizonte": F.lit(f"{horizonte['mes']}/{horizonte['ano']}"),
                      "mes_ano": F.lit(MES_ANO).cast("bigint")})
        .localCheckpoint())

    # 7 ── MÉTRICAS FINAIS ────────────────────────────────────────
    mf = res.agg(
        F.corr("Faturamento","previsoes").alias("corr"),
        F.sqrt(F.avg(F.pow(F.col("Faturamento")-F.col("previsoes"),2))).alias("rmse"),
        F.avg(F.abs(F.col("Faturamento")-F.col("previsoes"))).alias("mae"),
        F.sum("Faturamento").alias("fat_real"),
        F.sum("previsoes").alias("fat_prev")).first()
    r2f = (mf["corr"]**2) if mf["corr"] else float("nan")
    elapsed = _time.time() - t0
    _p(f"7/7 concluído em {elapsed:.1f}s — R²={r2f:.4f}  RMSE={mf['rmse']:.2f}  MAE={mf['mae']:.2f}")

    return res, {
        "horizonte": f"{horizonte['mes']}/{horizonte['ano']}",
        "r2_valid": r2v, "rmse_valid": mv["rmse"], "mae_valid": mv["mae"],
        "fat_real_valid": mv["fat_real"], "fat_prev_valid": mv["fat_prev"], "n_valid": mv["n"],
        "r2": r2f, "rmse": mf["rmse"], "mae": mf["mae"],
        "fat_real": mf["fat_real"], "fat_prev": mf["fat_prev"],
        "tempo_s": elapsed,
    }

In [0]:
# ============ EXECUÇÃO PARALELA DOS HORIZONTES ============
# Cada horizonte roda em uma thread; o Spark executa os jobs simultaneamente.
# Validado: métricas idênticas à execução sequencial, ~1.7x mais rápido.
from concurrent.futures import ThreadPoolExecutor, as_completed

NUM_WORKERS_XGB = 6                 # tasks por treino XGBoost (3 threads x 4 = 12 núcleos)
MAX_THREADS = len(HORIZONTES)       # reduza se faltar núcleo/memória no cluster

_t0 = _time.time()
resultados_horizontes = [None] * len(HORIZONTES)
metricas_horizontes = [None] * len(HORIZONTES)

with ThreadPoolExecutor(max_workers=MAX_THREADS) as pool:
    futuros = {
        pool.submit(processar_horizonte, i, h, dados_brasil, mes_map, VALID,
                    num_workers_xgb=NUM_WORKERS_XGB, verbose=True): i
        for i, h in enumerate(HORIZONTES)
    }
    for f in as_completed(futuros):
        i = futuros[f]
        # .result() repropaga qualquer erro ocorrido dentro da thread
        resultados_horizontes[i], metricas_horizontes[i] = f.result()

tempo_total = _time.time() - _t0

# Consolidar (usado pelas células seguintes de pós-processamento)
resultado_final = reduce(lambda x, y: x.unionByName(y, allowMissingColumns=True),
                         resultados_horizontes)

rotulos = ["M+1", "M+2", "M+3"]
linha = "=" * 104

print(f"\n{linha}")
print(f"EXECUÇÃO PARALELA: {tempo_total:.0f}s total ({MAX_THREADS} threads)")

# ---- VALIDAÇÃO: mês VALID com faturamento REAL (métricas confiáveis) ----
# Cada horizonte prevê o mesmo mês VALID, mas com os lags do seu horizonte
# (M+1 usa lag 2+, M+2 lag 3+, M+3 lag 4+). Considera apenas linhas não-zero (base do XGBoost).
print(f"\n{linha}")
print(f"VALIDAÇÃO — mês {VALID} (faturamento real)")
print(linha)
print(f"{'Horiz.':<7}{'Alvo':<10}{'Linhas':>9}{'R²':>9}{'RMSE':>10}{'MAE':>9}"
      f"{'Real':>18}{'Previsto':>18}{'Prev/Real':>11}")
for rot, m in zip(rotulos, metricas_horizontes):
    razao = m['fat_prev_valid'] / m['fat_real_valid'] if m['fat_real_valid'] else float('nan')
    print(f"{rot:<7}{m['horizonte']:<10}{m['n_valid']:>9,}{m['r2_valid']:>9.4f}{m['rmse_valid']:>10.1f}"
          f"{m['mae_valid']:>9.1f}{m['fat_real_valid']:>18,.0f}{m['fat_prev_valid']:>18,.0f}{razao:>11.2%}")

# ---- TESTE: meses futuros. O "real" é cópia do último mês (placeholder) ----
print(f"\n{linha}")
print("TESTE — meses futuros (o 'real' é cópia do último mês: métricas apenas indicativas)")
print(linha)
print(f"{'Horiz.':<7}{'Alvo':<10}{'R²':>9}{'RMSE':>10}{'MAE':>9}{'Placeholder':>18}{'Previsto':>18}{'Tempo':>8}")
for rot, m in zip(rotulos, metricas_horizontes):
    print(f"{rot:<7}{m['horizonte']:<10}{m['r2']:>9.4f}{m['rmse']:>10.1f}{m['mae']:>9.1f}"
          f"{m['fat_real']:>18,.0f}{m['fat_prev']:>18,.0f}{m['tempo_s']:>7.0f}s")
print(linha)

# Tabela de métricas como DataFrame (para consulta/exibição)
metricas_df = spark.createDataFrame([
    (rot, m["horizonte"], int(VALID), int(m["n_valid"]), float(m["r2_valid"]), float(m["rmse_valid"]),
     float(m["mae_valid"]), float(m["fat_real_valid"]), float(m["fat_prev_valid"]),
     float(m["r2"]), float(m["rmse"]), float(m["mae"]), float(m["fat_prev"]))
    for rot, m in zip(rotulos, metricas_horizontes)
], "horizonte string, mes_alvo string, mes_validacao int, linhas_validacao int, r2_validacao double, "
   "rmse_validacao double, mae_validacao double, real_validacao double, previsto_validacao double, "
   "r2_teste double, rmse_teste double, mae_teste double, previsto_teste double")
display(metricas_df)

In [0]:
# acrescentar data de processamento
from datetime import datetime

resultado_final = resultado_final.withColumn("data_processamento", F.lit(datetime.now().strftime("%Y-%m-%d %H:%M:%S")))

In [0]:
# Rateio: participação de media_volume_6m_Pedidos no grupo (mes_ano, Familia, Centro) x previsão total do grupo
# media_volume_6m_Pedidos está em escala log1p -> destransformar com expm1 (e^x - 1) antes do rateio
w_rateio = Window.partitionBy("mes_ano", "Familia", "Centro")
media_6m_ped_orig = F.expm1(F.col("media_volume_6m_Pedidos"))
resultado_final = resultado_final.withColumn(
    "previsao_rateada",
    # try_divide: grupo com soma 0 -> nulo (no pandas virava NaN); evita erro DIVIDE_BY_ZERO do modo ANSI
    F.try_divide(media_6m_ped_orig, F.sum(media_6m_ped_orig).over(w_rateio))
    * F.sum("previsoes").over(w_rateio)
)

In [0]:
display(resultado_final)

In [0]:
# select, rename e etc
# (tipos alinhados ao schema existente de desenvolvimento.bronze_demanda.tb_previsao_demanda)
resultado_final = resultado_final.withColumnsRenamed({
    "Sub_Canal": "nm_segmento",
    "BR": "ds_escritorio_vendas",
    "GR": "ds_equipe_vendas",
    "SKU": "cd_produto",
    "Centro": "cd_centro",
    "Familia": "nm_familia",
    "Ano": "nr_ano",
    "Mes": "nm_mes",
    "Data": "cd_anomes",
    "previsoes": "qt_previsao",
    "data_processamento": "dh_processamento",
    "previsao_rateada": "qt_previsao_rateada"})
resultado_final = resultado_final.select(
    "nm_segmento", "ds_escritorio_vendas", "ds_equipe_vendas", "cd_produto", "cd_centro", "nm_familia",
    F.col("nr_ano").cast("double").alias("nr_ano"),
    "nm_mes",
    F.col("cd_anomes").cast("double").alias("cd_anomes"),
    F.col("qt_previsao").cast("double").alias("qt_previsao"),
    F.col("qt_previsao_rateada").cast("double").alias("qt_previsao_rateada"),
    "dh_processamento"
)

In [0]:
display(resultado_final)

In [0]:
caminho = "desenvolvimento.gold_planejamento_integrado.tb_previsao_demanda"

# resultado_final já é um Spark DataFrame -> escrita distribuída direta
(resultado_final.write.format("delta").mode("overwrite").saveAsTable(caminho))
